# Setup — Registrar Modelos en Registry
**Ejecutar UNA VEZ** para pre-poblar el Model Registry con 2 modelos independientes.
Este notebook NO es para el participante — es solo para setup.

In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE CREDIBANCO_HOL;
USE WAREHOUSE CREDIBANCO_HOL_WH;
USE SCHEMA ANALITICA;

## Modelo 1: Detección de Fraude (XGBoost)

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

session = get_active_session()
reg = Registry(session, database_name='CREDIBANCO_HOL', schema_name='ANALITICA')

# Datos
train_pd = session.table('CREDIBANCO_HOL.ANALITICA.TRAIN_RIESGO_COMERCIO').to_pandas()
feature_cols = [c for c in train_pd.columns if c not in ['COMERCIO_ID', 'ES_FRAUDE']]
X = train_pd[feature_cols].fillna(0)
y = train_pd['ES_FRAUDE']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

neg, pos = int((y_train == 0).sum()), int((y_train == 1).sum())
spw = neg / pos if pos > 0 else 1

# V1: XGBoost base
model_v1 = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, scale_pos_weight=spw, random_state=42, eval_metric='logloss')
model_v1.fit(X_train, y_train)
y_pred = model_v1.predict(X_test)
y_proba = model_v1.predict_proba(X_test)[:, 1]
metrics_v1 = {'accuracy': accuracy_score(y_test, y_pred), 'precision': precision_score(y_test, y_pred, zero_division=0), 'recall': recall_score(y_test, y_pred, zero_division=0), 'f1': f1_score(y_test, y_pred, zero_division=0), 'auc_roc': roc_auc_score(y_test, y_proba)}

try:
    reg.delete_model('MODELO_FRAUDE_CREDIBANCO')
except:
    pass

mv1 = reg.log_model(model_v1, model_name='MODELO_FRAUDE_CREDIBANCO', version_name='V1', sample_input_data=X_train.head(10), metrics=metrics_v1, conda_dependencies=['xgboost', 'scikit-learn'], target_platforms=['WAREHOUSE'], comment='XGBoost fraude V1 base')
mv1.set_alias('champion')
print(f'V1 registrada: {metrics_v1}')

# V2: XGBoost tuned
model_v2 = XGBClassifier(n_estimators=200, max_depth=7, learning_rate=0.05, scale_pos_weight=spw, random_state=42, eval_metric='logloss')
model_v2.fit(X_train, y_train)
y_pred2 = model_v2.predict(X_test)
y_proba2 = model_v2.predict_proba(X_test)[:, 1]
metrics_v2 = {'accuracy': accuracy_score(y_test, y_pred2), 'precision': precision_score(y_test, y_pred2, zero_division=0), 'recall': recall_score(y_test, y_pred2, zero_division=0), 'f1': f1_score(y_test, y_pred2, zero_division=0), 'auc_roc': roc_auc_score(y_test, y_proba2)}

mv2 = reg.log_model(model_v2, model_name='MODELO_FRAUDE_CREDIBANCO', version_name='V2', sample_input_data=X_train.head(10), metrics=metrics_v2, conda_dependencies=['xgboost', 'scikit-learn'], target_platforms=['WAREHOUSE'], comment='XGBoost fraude V2 tuned')
mv2.set_alias('challenger')
print(f'V2 registrada: {metrics_v2}')

## Modelo 2: Predicción de Churn (XGBoost)

In [ ]:
# Modelo de churn independiente
churn_pd = session.table('CREDIBANCO_HOL.ANALITICA.COMERCIOS_CHURN').to_pandas()
feature_cols_churn = [c for c in churn_pd.columns if c not in ['COMERCIO_ID', 'ES_CHURN']]
Xc = churn_pd[feature_cols_churn].fillna(0)
yc = churn_pd['ES_CHURN']
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)

neg_c, pos_c = int((yc_train == 0).sum()), int((yc_train == 1).sum())
spw_c = neg_c / pos_c if pos_c > 0 else 1

model_churn = XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.08, scale_pos_weight=spw_c, random_state=42, eval_metric='logloss')
model_churn.fit(Xc_train, yc_train)
yc_pred = model_churn.predict(Xc_test)
yc_proba = model_churn.predict_proba(Xc_test)[:, 1]
metrics_churn = {'accuracy': accuracy_score(yc_test, yc_pred), 'precision': precision_score(yc_test, yc_pred, zero_division=0), 'recall': recall_score(yc_test, yc_pred, zero_division=0), 'f1': f1_score(yc_test, yc_pred, zero_division=0), 'auc_roc': roc_auc_score(yc_test, yc_proba)}

try:
    reg.delete_model('MODELO_CHURN_COMERCIOS')
except:
    pass

mv_churn = reg.log_model(model_churn, model_name='MODELO_CHURN_COMERCIOS', version_name='V1', sample_input_data=Xc_train.head(10), metrics=metrics_churn, conda_dependencies=['xgboost', 'scikit-learn'], target_platforms=['WAREHOUSE'], comment='XGBoost churn de comercios')
mv_churn.set_alias('champion')
print(f'Churn V1 registrada: {metrics_churn}')

In [ ]:
# Verificar
print('Modelos en Registry:')
for m in reg.list_models():
    print(f'  {m.name}: {[v.version_name for v in m.versions()]}')